In [1]:
import pandas as pd

from Pipeline.Global.GallstoneDataSet import GallstoneDataSet
from Pipeline.Methodology.EvaluationELM import EvaluationELM
from Pipeline.Global.GlobalSetting import GlobalSetting

In [2]:
gallstone_dataset = GallstoneDataSet()
gallstone_dataset.fetch_data_path_1()

features_size = gallstone_dataset.x_train.shape[1]

In [3]:
# 1. Instantiate the Evaluator Object ONCE
evaluator = EvaluationELM(
    x_train = gallstone_dataset.x_train,
    y_train = gallstone_dataset.y_train,
    activation_function     = GlobalSetting.sigmoid ,
    elm_init_seed_range     = GlobalSetting.elm_initial_state_range ,
    k_fold = 5
)

In [4]:
hidden_size_record ,hidden_size_result = evaluator.grid_search_hidden_size(
    GlobalSetting.hidden_size_explore_range
)
GlobalSetting.save_dataframe_to_record(hidden_size_record,'hidden_size_record.csv')
hidden_node_list = EvaluationELM.extract_top_results(hidden_size_result)
best_hidden_size = hidden_node_list.iloc[0]

[I/O Trace] Record exported successfully: ../../Storage/Record\hidden_size_record.csv


In [5]:
lambda_record, lambda_result = evaluator.grid_search_lambda(
    hidden_size     = best_hidden_size['Hidden_Nodes'],
    lambda_range    = GlobalSetting.lambda_value_explore_range
)
GlobalSetting.save_dataframe_to_record(lambda_record,'lambda_record.csv')
lambda_value_list = EvaluationELM.extract_top_results(lambda_result)
best_lambda_value = lambda_value_list.iloc[0]

[I/O Trace] Record exported successfully: ../../Storage/Record\lambda_record.csv


In [6]:
size_lambda_record , size_lambda_result = evaluator.grid_search_hidden_size_and_lambda(
    hidden_size_range   = hidden_node_list['Hidden_Nodes'],
    lambda_range        = GlobalSetting.lambda_value_explore_range
)
GlobalSetting.save_dataframe_to_record(size_lambda_record,'size_lambda_record.csv')
size_lambda_list = EvaluationELM.extract_top_results(size_lambda_result)
best_size_lambda = size_lambda_list.iloc[0]

[I/O Trace] Record exported successfully: ../../Storage/Record\size_lambda_record.csv


In [7]:
model_configs_payload = [
    {
        "Model_Types" : "Best_Hidden_Nodes",
        "Hidden_Nodes": int(best_hidden_size['Hidden_Nodes']),
        "Activation": "sigmoid",
        "Lambda_Value": float(best_hidden_size['Lambda_Value'])
    },
    {
        "Model_Types" : "Best_Lambda",
        "Hidden_Nodes": int(best_lambda_value['Hidden_Nodes']),
        "Activation": "sigmoid",
        "Lambda_Value": float(best_lambda_value['Lambda_Value'])
    },
    {
        "Model_Types" : "Best_Size_and_Lambda",
        "Hidden_Nodes": int(best_size_lambda['Hidden_Nodes']),
        "Activation": "sigmoid",
        "Lambda_Value": float(best_size_lambda['Lambda_Value'])
    }
]
GlobalSetting.upsert_model_configs(model_configs_payload)

In [8]:
combined_df = pd.concat([best_hidden_size, best_lambda_value, best_size_lambda], axis=1)
combined_df

,47,20,20
Hidden_Nodes,48,48,48
Activation,sigmoid,sigmoid,sigmoid
Lambda_Value,0.0,0.03125,0.03125
avg_Accuracy_Seed_Mean,0.726667,0.735033,0.735033
avg_Accuracy_Seed_Std,0.020264,0.017254,0.017254
avg_Accuracy_Seed_SEM,0.0037,0.00315,0.00315
avg_Accuracy_Seed_Min,0.666667,0.709804,0.709804
avg_Accuracy_Seed_Max,0.760784,0.768627,0.768627
avg_Precision_Seed_Mean,0.761207,0.777281,0.777281
avg_Precision_Seed_Std,0.026807,0.022277,0.022277
